# SpaceX Falcon 9 Web Scraping


This notebook analyzes SpaceX Falcon 9 launch data to explore factors affecting first-stage landing success and build predictive models.


## Objective
- Parse Falcon 9 and Falcon Heavy launch history from a fixed Wikipedia snapshot.
- Extract structured launch attributes from the HTML tables.
- Save the scraped output to `data/spacex_web_scraped.csv`.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "data").exists() and (candidate / "notebooks").exists()
)
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

import unicodedata

import pandas as pd
from bs4 import BeautifulSoup

from notebook_utils import data_path, ensure_text_file


## Data Loading


In [2]:
STATIC_URL = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
html_path = ensure_text_file("falcon9_wiki_snapshot.html", STATIC_URL)
soup = BeautifulSoup(html_path.read_text(encoding="utf-8"), "html.parser")
soup.title.string


'List of Falcon 9 and Falcon Heavy launches - Wikipedia'

## Data Cleaning


In [3]:
def date_time(table_cells):
    return [value.strip() for value in list(table_cells.strings)][0:2]


def booster_version(table_cells):
    values = [value for index, value in enumerate(table_cells.strings) if index % 2 == 0][0:-1]
    return "".join(values)


def landing_status(table_cells):
    return list(table_cells.strings)[0]


def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    return mass[0 : mass.find("kg") + 2] if mass and "kg" in mass else "0 kg"


def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = " ".join(row.contents)
    if not column_name.strip().isdigit():
        return column_name.strip()
    return None


def get_link_text(cell):
    if cell.a and cell.a.string:
        return cell.a.string.strip()
    return cell.get_text(" ", strip=True)


In [4]:
html_tables = soup.find_all("table")
first_launch_table = html_tables[2]
column_names = []
for row in first_launch_table.find_all("th"):
    name = extract_column_from_header(row)
    if name:
        column_names.append(name)
column_names


['Flight No.',
 'Date and time ( )',
 'Launch site',
 'Payload',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launch outcome']

## Feature Engineering


In [5]:
launch_dict = dict.fromkeys(column_names)
del launch_dict["Date and time ( )"]
launch_dict.update(
    {
        "Flight No.": [],
        "Launch site": [],
        "Payload": [],
        "Payload mass": [],
        "Orbit": [],
        "Customer": [],
        "Launch outcome": [],
        "Version Booster": [],
        "Booster landing": [],
        "Date": [],
        "Time": [],
    }
)

for table in soup.find_all("table", "wikitable plainrowheaders collapsible"):
    for row in table.find_all("tr"):
        if not row.th or not row.th.string:
            continue
        flight_number = row.th.string.strip()
        if not flight_number.isdigit():
            continue

        cells = row.find_all("td")
        datatime = date_time(cells[0])
        launch_dict["Flight No."].append(flight_number)
        launch_dict["Date"].append(datatime[0].strip(","))
        launch_dict["Time"].append(datatime[1])
        launch_dict["Version Booster"].append(booster_version(cells[1]) or get_link_text(cells[1]))
        launch_dict["Launch site"].append(get_link_text(cells[2]))
        launch_dict["Payload"].append(get_link_text(cells[3]))
        launch_dict["Payload mass"].append(get_mass(cells[4]))
        launch_dict["Orbit"].append(get_link_text(cells[5]))
        launch_dict["Customer"].append(get_link_text(cells[6]))
        launch_dict["Launch outcome"].append(list(cells[7].strings)[0].strip())
        launch_dict["Booster landing"].append(landing_status(cells[8]).strip())

df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.head()


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0 kg,LEO,SpaceX,Success,F9 v1.07B0003.18,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0 kg,LEO,NASA,Success,F9 v1.07B0004.18,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.18,No attempt,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.18,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.18,No attempt,1 March 2013,15:10


In [6]:
output_path = data_path("spacex_web_scraped.csv")
df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/riyana/Desktop/spacex/data/spacex_web_scraped.csv')

## Key Findings


In [7]:
df["Launch site"].value_counts().head()


Launch site
CCAFS             40
KSC               33
Cape Canaveral    20
VAFB              16
CCSFS             12
Name: count, dtype: int64